# 🐾 Review & Select Representative Images for iWildCam Classes

This interactive notebook lets you quickly inspect and approve the representative camera-trap photo for each of the 182 classes.

### How to use:
1. **Inspect:** Look at the displayed photo for the current class.
2. **If it looks good:** Click **`✅ Approve & Next Class`** (green button).
3. **If it is blurry / dark / occluded:** Click **`🔄 Try Another Image`** to cycle through other photos of that species.
4. **Resume anytime:** Your approvals are automatically saved to `phase1/pair_a/data/representative_images.json`. The Dash dashboard (`species_confusion_network.py`) will automatically use your approved photos!

In [ ]:
from pathlib import Path
import csv
import json
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# Paths (resolves properly whether run from repo root or notebooks folder)
DATA_DIR = Path("data/iwildcam_v2.0/train")
META_CSV = Path("data/iwildcam_v2.0/metadata.csv")
CAT_CSV = Path("data/iwildcam_v2.0/categories.csv")
OUTPUT_JSON = Path("phase1/pair_a/data/representative_images.json")

# If working directory is inside notebooks/, look three levels up
if not DATA_DIR.is_dir() and Path("../../../data/iwildcam_v2.0/train").is_dir():
    DATA_DIR = Path("../../../data/iwildcam_v2.0/train")
    META_CSV = Path("../../../data/iwildcam_v2.0/metadata.csv")
    CAT_CSV = Path("../../../data/iwildcam_v2.0/categories.csv")
    OUTPUT_JSON = Path("../data/representative_images.json")

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Output JSON destination:", OUTPUT_JSON.resolve())
assert DATA_DIR.is_dir(), f"Data directory not found at {DATA_DIR}"
assert META_CSV.is_file(), f"Metadata CSV not found at {META_CSV}"

In [ ]:
# Load categories and candidates
categories = {}
with open(CAT_CSV, mode="r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        categories[int(row["y"])] = row["name"]

print("Scanning metadata to collect candidate photos for all 182 classes...")
candidates = {y: [] for y in categories}
with open(META_CSV, mode="r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        y = int(row["y"])
        if y in candidates and len(candidates[y]) < 30:
            fn = row["filename"]
            if (DATA_DIR / fn).is_file():
                candidates[y].append({
                    "filename": fn,
                    "location": row.get("location", ""),
                    "datetime": row.get("datetime", "")
                })

# Load previously approved selections if already saved
approved = {}
if OUTPUT_JSON.is_file():
    try:
        with open(OUTPUT_JSON, "r") as f:
            approved = {int(k): v for k, v in json.load(f).items()}
        print(f"Loaded {len(approved)} previously approved classes from {OUTPUT_JSON}")
    except Exception as e:
        print("Error loading saved selections:", e)
else:
    print("Starting fresh review (no existing approvals file yet).")


In [ ]:
# Interactive Review Widget
class_ids = sorted(list(categories.keys()))
current_state = {
    "class_idx": 0,
    "cand_idx": 0,
}

# Jump to the first unapproved class
for i, cid in enumerate(class_ids):
    if cid not in approved:
        current_state["class_idx"] = i
        break

def save_to_disk():
    with open(OUTPUT_JSON, "w") as f:
        json.dump(approved, f, indent=2)

# UI Components
class_dropdown = widgets.Dropdown(
    options=[(f"{categories[cid]} [ID: {cid}]", idx)
             for idx, cid in enumerate(class_ids)],
    value=current_state["class_idx"],
    description="Select Class:",
    layout=widgets.Layout(width="480px")
)

status_label = widgets.HTML()
img_output = widgets.Output()
notification_label = widgets.HTML()

btn_approve = widgets.Button(
    description="✅ Approve & Next Class",
    button_style="success",
    layout=widgets.Layout(width="210px", height="36px", font_weight="bold")
)
btn_next_cand = widgets.Button(
    description="🔄 Try Another Image",
    button_style="warning",
    layout=widgets.Layout(width="180px", height="36px")
)
btn_prev_cand = widgets.Button(
    description="⏮️ Prev Image",
    layout=widgets.Layout(width="120px", height="36px")
)
btn_skip = widgets.Button(
    description="⏭️ Skip Class",
    layout=widgets.Layout(width="120px", height="36px")
)
btn_prev_class = widgets.Button(
    description="⬅️ Prev Class",
    layout=widgets.Layout(width="120px", height="36px")
)

def update_display():
    c_idx = current_state["class_idx"]
    cid = class_ids[c_idx]
    cand_idx = current_state["cand_idx"]
    cands = candidates.get(cid, [])
    c_name = categories.get(cid, "Unknown")
    
    is_appr = cid in approved
    if is_appr:
        appr_badge = f"<span style='background:#dcfce7;color:#15803d;padding:3px 8px;border-radius:4px;font-weight:bold;'>✓ APPROVED ({approved[cid]})</span>"
    else:
        appr_badge = "<span style='background:#fef3c7;color:#b45309;padding:3px 8px;border-radius:4px;'>Not yet approved</span>"

    status_label.value = (
        f"<div style='font-family: sans-serif; font-size: 14px; margin: 6px 0 10px 0;'>"
        f"<b>Progress:</b> {len(approved)} / {len(class_ids)} approved &nbsp;|&nbsp; "
        f"<b>Candidate:</b> {cand_idx + 1} of {len(cands)} &nbsp;|&nbsp; {appr_badge}"
        f"</div>"
    )

    class_dropdown.value = c_idx

    with img_output:
        clear_output(wait=True)
        if not cands:
            print("No candidate images found for this class in train directory.")
            return
        cand = cands[cand_idx]
        p = DATA_DIR / cand["filename"]
        if p.is_file():
            img = Image.open(p)
            img.thumbnail((740, 480))
            display(img)
            loc_info = f"Camera #{cand['location']}" if cand['location'] else "Unknown camera"
            dt_info = f" | Date: {cand['datetime']}" if cand['datetime'] else ""
            print(f"{c_name} [Class ID: {cid}]")
            print(f"File: {cand['filename']} ({loc_info}{dt_info})")
            if is_appr and approved[cid] == cand["filename"]:
                print("★ This is your currently APPROVED photo for this species.")
        else:
            print(f"File not found: {p}")

# Callbacks
def on_approve(b):
    c_idx = current_state["class_idx"]
    cid = class_ids[c_idx]
    cands = candidates.get(cid, [])
    if cands:
        chosen = cands[current_state["cand_idx"]]["filename"]
        approved[cid] = chosen
        save_to_disk()
        notification_label.value = f"<span style='color:#16a34a;font-weight:600;'>✓ Saved {categories[cid]}!</span>"
    if c_idx < len(class_ids) - 1:
        current_state["class_idx"] += 1
        current_state["cand_idx"] = 0
    update_display()

def on_next_cand(b):
    cid = class_ids[current_state["class_idx"]]
    cands = candidates.get(cid, [])
    if cands:
        current_state["cand_idx"] = (current_state["cand_idx"] + 1) % len(cands)
        notification_label.value = ""
    update_display()

def on_prev_cand(b):
    cid = class_ids[current_state["class_idx"]]
    cands = candidates.get(cid, [])
    if cands:
        current_state["cand_idx"] = (current_state["cand_idx"] - 1) % len(cands)
        notification_label.value = ""
    update_display()

def on_skip(b):
    if current_state["class_idx"] < len(class_ids) - 1:
        current_state["class_idx"] += 1
        current_state["cand_idx"] = 0
        notification_label.value = ""
    update_display()

def on_prev_class(b):
    if current_state["class_idx"] > 0:
        current_state["class_idx"] -= 1
        current_state["cand_idx"] = 0
        notification_label.value = ""
    update_display()

def on_dropdown_change(change):
    if change.get("type") == "change" and change.get("name") == "value":
        current_state["class_idx"] = change["new"]
        current_state["cand_idx"] = 0
        notification_label.value = ""
        update_display()

btn_approve.on_click(on_approve)
btn_next_cand.on_click(on_next_cand)
btn_prev_cand.on_click(on_prev_cand)
btn_prev_class.on_click(on_prev_class)
btn_skip.on_click(on_skip)
class_dropdown.observe(on_dropdown_change)

display(widgets.VBox([
    widgets.HBox([class_dropdown, notification_label]),
    status_label,
    widgets.HBox([btn_approve, btn_next_cand, btn_prev_cand, btn_prev_class, btn_skip]),
    img_output
]))
update_display()


## 📊 Summary of Approved Classes
Run the cell below to inspect a table of all classes you have reviewed so far.

In [ ]:
import pandas as pd
if OUTPUT_JSON.is_file():
    with open(OUTPUT_JSON, "r") as f:
        data = json.load(f)
    df_summary = pd.DataFrame([{
        "Class ID": int(k),
        "Species Name": categories.get(int(k), ""),
        "Approved Image": v
    } for k, v in data.items()]).sort_values("Class ID")
    print(f"Approved: {len(df_summary)} / {len(categories)} classes")
    display(df_summary)
else:
    print("No approved images saved yet.")
